# Week 2: Embeddings

**Goal**: How does a neural network understand that "Receive" and "Restock" are more similar than "Receive" and "Scenario"?

Last week, we built a very simple neural network that learned to predict the next word in a sequence using **One-Hot Encoding**. The limitation of one-hot encoding is that every word is treated as completely independent. 

In this notebook, we will modify the network to learn an **Embedding** for each word. An embedding is a dense vector of numbers (e.g., `[0.72, -0.15, 0.44]`) where the mathematical distance between vectors captures the semantic similarity between the words.

## 1. Imports
We need `numpy` for our neural network math, `matplotlib` for plotting our results, and `PCA` from `scikit-learn` to squash our high-dimensional embeddings down to 2D for visualization.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

## 2. Vocabulary and Training Data
We are using the extended vocabulary which includes operational words (Receive, Restock) and inventory words (Cabinet, Shelf, Bin).

In [ ]:
words = [
    "Inventory", "Order", "Cabinet", "Drug", "Invoice", 
    "Shipment", "Receive", "Restock", "Forecast", "Scenario",
    "Acknowledge", "Shelf", "Bin"
]

# Create Tokenizer mapping
vocab = {word: i for i, word in enumerate(words)}
id_to_vocab = {i: word for word, i in vocab.items()}
vocab_size = len(vocab)

print(f"Vocabulary Size: {vocab_size}")

# Our sequential training data
training_data = [
    (vocab["Order"], vocab["Shipment"]),
    (vocab["Shipment"], vocab["Receive"]),
    (vocab["Receive"], vocab["Restock"]),
    (vocab["Restock"], vocab["Inventory"]),
    (vocab["Inventory"], vocab["Forecast"]),
    (vocab["Forecast"], vocab["Order"])
]

## 3. Initializing the Neural Network
Instead of `W1`, we now have an **Embedding Matrix** (`W_embed`). We will use 8 dimensions for our embeddings. 
Notice that the shape is `(vocab_size, embedding_dim)`.

In [ ]:
np.random.seed(42) # For reproducible results

embedding_dim = 8
learning_rate = 0.1

# The Embedding Matrix (replaces W1)
W_embed = np.random.randn(vocab_size, embedding_dim)

# The Output Weights (replaces W2)
W_out = np.random.randn(embedding_dim, vocab_size)
b_out = np.zeros((1, vocab_size))

## 4. The Training Loop
Watch closely at the **Forward Pass**. We completely removed `one_hot` and matrix multiplication for the first layer. Instead, we just do a direct array index lookup: `embedding = W_embed[x]`.

We also calculate the gradient for the embedding row and update it using backpropagation.

In [ ]:
epochs = 1000

for epoch in range(epochs):
    total_loss = 0
    
    for x, y in training_data:
        # --- FORWARD PASS ---
        # 1. Embedding Lookup (Highly efficient $O(1)$ replacement for One-Hot Matrix Multiplication)
        # We add np.newaxis to keep it as a 2D matrix shape (1, embedding_dim)
        embedding = W_embed[x][np.newaxis, :]
        
        # 2. Linear Transformation & Logits
        logits = embedding @ W_out + b_out
        
        # 3. Softmax
        exp_logits = np.exp(logits - np.max(logits))
        probs = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)
        
        # --- LOSS CALCULATION ---
        loss = -np.log(probs[0, y] + 1e-9)
        total_loss += loss
        
        # --- BACKWARD PASS ---
        # 1. Gradient at Output
        d_logits = probs.copy()
        d_logits[0, y] -= 1
        
        # 2. Gradients for Output Layer
        dW_out = embedding.T @ d_logits
        db_out = d_logits
        
        # 3. Gradient flowing back to the Embedding Layer
        d_embedding = d_logits @ W_out.T
        
        # --- PARAMETER UPDATE ---
        W_out -= learning_rate * dW_out
        b_out -= learning_rate * db_out
        
        # We ONLY update the specific row of the embedding matrix that was used!
        W_embed[x] -= learning_rate * d_embedding[0]

    if (epoch + 1) % 200 == 0:
        print(f"Epoch {epoch + 1} | Loss: {total_loss:.4f}")


## 5. Cosine Similarity
Now that training is complete, let's see if the network learned that "Receive" and "Restock" are related concepts. We will calculate the Cosine Similarity between their learned embedding vectors.

In [ ]:
def cosine_similarity(v1, v2):
    dot_product = np.dot(v1, v2)
    magnitude = np.linalg.norm(v1) * np.linalg.norm(v2)
    return dot_product / (magnitude + 1e-9)

emb_receive = W_embed[vocab["Receive"]]
emb_restock = W_embed[vocab["Restock"]]
emb_forecast = W_embed[vocab["Forecast"]]

print(f"Similarity (Receive vs Restock) : {cosine_similarity(emb_receive, emb_restock):.4f}")
print(f"Similarity (Receive vs Forecast): {cosine_similarity(emb_receive, emb_forecast):.4f}")

## 6. Dimensionality Reduction & Visualization
Our embeddings have 8 dimensions, meaning they live in 8D space. We can't plot that on a computer screen. 
We will use **PCA (Principal Component Analysis)** to squash the 8 dimensions down to 2 dimensions so we can visualize how the words clustered.

In [ ]:
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(W_embed)

plt.figure(figsize=(10, 8))
plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], color='blue', s=100)

# Add labels to the points
for i, word in id_to_vocab.items():
    plt.annotate(word, (embeddings_2d[i, 0], embeddings_2d[i, 1]), 
                 xytext=(5, 5), textcoords='offset points', fontsize=12)

plt.title("2D PCA Visualization of Learned Word Embeddings")
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

> **Exercise/Stretch Goal**: Look at the plot. Did the sequential words form a trajectory or cluster together? What happens to the un-trained words like "Cabinet" or "Bin"? Why are they scattered randomly?